# Verification and Self-Correction

Self-correction — asking a model to review and revise its own output — sounds promising but has a well-documented failure mode: models often reinforce their original mistakes rather than catching them. Understanding when self-correction helps and when it hurts is essential for building reliable reasoning systems.

## The Self-Correction Problem

Huang et al. (2023) 'Large Language Models Cannot Self-Correct Reasoning Yet' showed that without external feedback, LLM self-correction degrades accuracy on reasoning tasks. The model's correction is biased toward its original answer.

Why this happens:
- The model generating the critique and the model that made the mistake are the same model — they share the same biases and knowledge gaps
- The model is asked to find errors in its own output, which triggers anchoring on the original content
- Critique generation tends to find surface-level issues (formatting, tone) rather than the substantive reasoning errors

Self-correction *does* work when:
1. External feedback is provided (execution results, test failures, retrieval evidence)
2. The correction prompt is carefully designed to force specific types of checks
3. A separate, larger model is used as the verifier

In [ ]:
from dataclasses import dataclass
from typing import Callable, Optional
import random

@dataclass
class VerificationResult:
    original: str
    critique: str
    revised: str
    original_correct: bool
    revised_correct: bool
    improved: bool
    degraded: bool

def verify_then_refine(
    problem: str,
    initial_response: str,
    verifier_fn: Callable,
    refine_fn: Callable,
    correct_answer: str,
    max_rounds: int = 2
) -> list:
    rounds = []
    current = initial_response
    for i in range(max_rounds):
        critique = verifier_fn(problem, current)
        revised = refine_fn(problem, current, critique)
        orig_correct = correct_answer in current
        rev_correct = correct_answer in revised
        rounds.append(VerificationResult(
            original=current[:100], critique=critique[:100], revised=revised[:100],
            original_correct=orig_correct, revised_correct=rev_correct,
            improved=not orig_correct and rev_correct,
            degraded=orig_correct and not rev_correct
        ))
        current = revised
    return rounds

rng = random.Random(42)

def blind_verifier(problem: str, response: str) -> str:
    # Without external knowledge, verifier often misses the actual error
    if rng.random() < 0.4:
        return 'The reasoning seems correct. The steps follow logically.'
    return 'There may be an arithmetic error. Please double-check the calculation.'

def blind_refiner(problem: str, response: str, critique: str) -> str:
    if 'arithmetic error' in critique:
        # Tends to introduce new errors while 'fixing'
        return response.replace('60', '65') if '60' in response else response + ' (revised)'
    return response  # leaves it unchanged if no specific issue

def grounded_verifier(problem: str, response: str) -> str:
    # Verifier with access to execution/lookup
    import re
    arith = re.search(r'(\d+)\s*([+\-*/])\s*(\d+)\s*=\s*(\d+)', response)
    if arith:
        a, op, b, claimed = float(arith.group(1)), arith.group(2), float(arith.group(3)), float(arith.group(4))
        ops = {'+': a+b, '-': a-b, '*': a*b, '/': a/b if b else None}
        expected = ops.get(op)
        if expected and abs(expected - claimed) > 0.01:
            return f'Arithmetic error: {a} {op} {b} = {expected}, not {claimed}'
    return 'No arithmetic errors found. Answer looks correct.'

def grounded_refiner(problem: str, response: str, critique: str) -> str:
    import re
    if 'Arithmetic error' in critique:
        # Targeted fix based on specific error message
        error_match = re.search(r'(\d+\.?\d*)\s*([+\-*/])\s*(\d+\.?\d*)\s*=\s*(\d+\.?\d*), not (\d+\.?\d*)', critique)
        if error_match:
            wrong = error_match.group(5)
            right = error_match.group(4)
            return response.replace(wrong, right)
    return response

problem = 'What is 15% of 80?'
bad_response = 'To find 15% of 80: 80 * 0.15 = 13.0. The answer is 13.0.'
correct = '12'

print('=== Blind self-correction ===')
rounds = verify_then_refine(problem, bad_response, blind_verifier, blind_refiner, correct)
for i, r in enumerate(rounds):
    print(f'Round {i+1}: correct={r.revised_correct}, improved={r.improved}, degraded={r.degraded}')
    print(f'  Critique: {r.critique[:80]}')

print('\n=== Grounded verification ===')
rounds2 = verify_then_refine(problem, bad_response, grounded_verifier, grounded_refiner, correct)
for i, r in enumerate(rounds2):
    print(f'Round {i+1}: correct={r.revised_correct}, improved={r.improved}')
    print(f'  Critique: {r.critique[:80]}')
    print(f'  Revised: {r.revised[:80]}')

## External Feedback as the Key

Verification works reliably when the feedback comes from an external oracle:

**Code execution**: run the generated code, pass the test results back as feedback. Extremely reliable signal — the execution environment is ground truth.

**Retrieval augmentation**: after generating an answer, retrieve relevant documents and check for factual consistency. Flags hallucinations that the model cannot self-detect.

**Formal verification**: for math proofs, compile in Lean/Isabelle. For logical claims, run a SAT/SMT solver. Provides unambiguous correct/incorrect feedback.

**Unit tests**: for coding tasks, write unit tests before generating code, then use test failures as precise feedback signals.

The pattern: give the model a clear, specific error message from an external system. 'Your code raised IndexError on line 4' is far more actionable than 'Your response may have errors, please review.'

## Iterative Refinement Best Practices

1. **Limit refinement rounds**: 1-2 rounds is usually optimal; more rounds rarely help and often hurt
2. **Use external feedback**: execution results, test failures, retrieval evidence
3. **Track improvement rate**: if accuracy doesn't improve by round 2, stop
4. **Don't self-critique without structure**: give the verifier a specific checklist (arithmetic, logical validity, factual accuracy), not an open-ended 'review this'
5. **Separate verifier and generator when possible**: a larger or different model as verifier reduces anchoring bias